In [ ]:
import torch
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))
from dimenet_clip import DimeNetBackbone, DimeNetReadout, DimeNetCLIP
from torch_geometric.data import Data, Batch
import torch.nn.functional as F
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score


In [ ]:
with open(POCKET_PKL, 'rb') as f:
    protein_vecs = pickle.load(f)
with open(LIGAND_PKL, 'rb') as f:
    ligand_vecs = pickle.load(f)


In [ ]:
with open(TARGET_LIST_PKL, 'rb') as f:
    lig_ids = pickle.load(f)


In [ ]:
def load_ddp_state_dict(model_path):
    state_dict = torch.load(model_path, map_location=torch.device('cpu'))
    updated_state_dict = {}
    for key in state_dict.keys():
        new_key = key[len('module.'): ] if key.startswith('module.') else key
        updated_state_dict[new_key] = state_dict[key]
    return updated_state_dict

In [ ]:
model = DimeNetCLIP(
    hidden_channels=128,
    out_channels=128,
    num_blocks=6,
    use_conformers=False,
)
state_dict = load_ddp_state_dict(
    os.path.join(MODEL_DIR, f"dimenet_clip_epoch_{MODEL_EPOCH}.pth")
)
model.load_state_dict(state_dict=state_dict)
model.to('cuda')
model.eval()


In [ ]:
targets = list(ligand_vecs.keys())

In [ ]:
ligand_vecs.keys()

In [ ]:
def batch_encode_ligands(ligand_data, batch_size=32, device='cuda'):
    n_chunks = int(np.ceil(len(ligand_data)/batch_size))
    chunk_ids = np.array_split(range(len(ligand_data)), n_chunks)
    encoded = []
    labels = []
    for ids in chunk_ids:
        data_list = []
        for idx in ids:
            z, pos, label = ligand_data[idx]
            z = torch.tensor(z)
            pos = torch.tensor(pos, dtype=torch.float)
            lig_data = Data(z=z, pos=pos)
            data_list.append(lig_data)
            labels.append(label)
        # 2. Batch them
        batch = Batch.from_data_list(data_list)
        batch.to(device)
        lig_emb = model.encode_ligand(batch.z, batch.pos, batch.batch)
        encoded.append(lig_emb)
    labels = torch.tensor(labels)
    return torch.cat(encoded), labels


In [ ]:
def calculate_ef(labels, scores, fraction=0.01):
    # Sort labels by score in descending order
    combined = sorted(zip(scores, labels), reverse=True, key=lambda x: x[0])
    sorted_labels = [label for score, label in combined]
    
    # Determine the number of ligands in the top x%
    n_total = len(sorted_labels)
    n_x = int(n_total * fraction)
    if n_x == 0: n_x = 1  # Ensure at least one ligand is considered
    
    # Calculate components
    hits_x = sum(sorted_labels[:n_x])
    hits_total = sum(sorted_labels)
    
    if hits_total == 0: return 0
    
    # Enrichment Factor calculation
    return (hits_x / n_x) / (hits_total / n_total)

In [ ]:
with torch.no_grad():
    prot_encoded = {}
    lig_encoded = {}
    for t in tqdm(targets):
        prot_encoded[t] = {}
        for pdbid, (z_prot, prot_coord) in protein_vecs[t].items():
            z_prot = torch.tensor(z_prot)
            prot_coord = torch.tensor(prot_coord, dtype=torch.float)
            prot_data = Data(z=z_prot, pos=prot_coord).to('cuda')
            batch = Batch.from_data_list([prot_data])
            prot_emb = model.encode_pocket(prot_data.z, prot_data.pos, batch.batch)
            prot_encoded[t][pdbid] = prot_emb.cpu()

        lig_encoded[t] = {}
        lig_data = list(ligand_vecs[t].values())
        cur_encoded, labels = batch_encode_ligands(lig_data)
        lig_encoded[t] = {'emb': cur_encoded.cpu(), 'labels': labels}

In [ ]:
preds = {}
for t in targets:
    preds[t] = {}
    lig_encoding = lig_encoded[t]['emb']
    labels = lig_encoded[t]['labels']
    for pdbid, prot_encoding in  prot_encoded[t].items():
        pred = torch.matmul(lig_encoding.to('cuda'), prot_encoding.to('cuda').T).flatten()
        # remove NaN 
        filtered_ids = torch.where(pred.flatten().isfinite())[0].cpu()
        filtered_labels = labels[filtered_ids]
        filtered_pred = pred[filtered_ids]
        ef1 = calculate_ef(filtered_labels, filtered_pred.cpu())
        auc = roc_auc_score(filtered_labels, filtered_pred.cpu())
        preds[t][pdbid] = {'auc': auc, 'ef1': ef1, 'preds': filtered_pred.cpu(), 'labels': filtered_labels}

In [ ]:
def make_predictions(lig_encoded, prot_encoded):
    preds = {}
    lig_encoding = lig_encoded['emb']
    labels = lig_encoded['labels']
    for pdbid, prot_encoding in  prot_encoded.items():
        pred = torch.matmul(lig_encoding.to('cuda'), prot_encoding.to('cuda').T).flatten()
        # remove NaN 
        filtered_ids = torch.where(pred.flatten().isfinite())[0].cpu()
        filtered_labels = labels[filtered_ids]
        filtered_pred = pred[filtered_ids]
        ef1 = calculate_ef(filtered_labels, filtered_pred.cpu())
        auc = roc_auc_score(filtered_labels, filtered_pred.cpu())
        preds[pdbid] = {'auc': auc, 'ef1': ef1, 'preds': filtered_pred.cpu(), 'labels': filtered_labels}
    return preds

In [ ]:
def report_targets(preds):
    all_aucs = []
    all_ef1s = []
    for t, by_pdbid in preds.items():
        print(t)
        aucs = []
        ef1s = []
        for pdbid, r in by_pdbid.items():
            aucs.append(r['auc'])
            ef1s.append(r['ef1'])
            
            # print(pdbid, round(r['auc'],3))
        n_ligs = r['labels'].shape[0]
        aucs = np.array(aucs)
        ef1s = np.array(ef1s)
        all_aucs.append(aucs.mean())
        all_ef1s.append(ef1s.mean())
        print(f'Num Structures: {aucs.shape[0]}, Num Ligands: {n_ligs}')
        print(f'AUC mean: {aucs.mean():.3f}, std:{aucs.std():.3f}, max: {aucs.max():.3f}, min: {aucs.min():.3f}')
        print(f'EF1 mean: {ef1s.mean():.3f}, std:{ef1s.std():.3f}, max: {ef1s.max():.3f}, min: {ef1s.min():.3f}')
    all_aucs = np.array(all_aucs)
    all_ef1s = np.array(all_ef1s)
    print("Mean AUCs", all_aucs.mean(), all_aucs.std())
    print("Mean EF1s", all_ef1s.mean(), all_ef1s.std())

In [ ]:
with open('pocket_encoded.pkl', 'wb') as p:
    pickle.dump(prot_encoded, p)
with open('ligand_encoded.pkl', 'wb') as p:
    pickle.dump(lig_encoded, p)

In [ ]:
# Resume from the cache written above instead of re-encoding.
with open('pocket_encoded.pkl', 'rb') as p:
    prot_encoded = pickle.load(p)
with open('ligand_encoded.pkl', 'rb') as p:
    lig_encoded = pickle.load(p)


In [ ]:
preds = {}
for t in targets:
    preds[t] = make_predictions(lig_encoded[t], prot_encoded[t])

In [ ]:
# reported model: PDBBind epoch 3 (val loss 22.98), fine-tuned from SAIR epoch 5; see weights/PROVENANCE.md
report_targets(preds)

In [ ]:
def get_recall(preds, labels):
    all_preds = preds.flatten()
    all_labels = labels.flatten()
    idx = np.argsort(all_preds).flip(0)

    recall = np.cumsum(all_labels[idx])
    return recall

def plot_roc_curve2(preds, labels, target_name):
    n_pos = labels.sum()
    recall = get_recall(preds, labels)
    plt.subplots(figsize=(8,8))
    plt.plot(recall, marker='.')
    sns.lineplot(
        x=range(len(labels)), 
        y=np.linspace(n_pos/len(labels), n_pos, len(labels)),
        label='Chance Level (AUC=0.5)', 
        linestyle='--', color='black', 
        alpha=0.5
    )
    plt.xlabel('Number of top predictions')
    plt.ylabel('Cumulative number of actives')
    plt.title(f'Recall Curve for {target_name}')
    plt.grid()
    plt.show()
    


In [ ]:
for t in targets:
    r = list(preds[t].values())[0]
    if len(r) == 0:
        continue
    plot_roc_curve2(r['preds'], r['labels'], t)

In [ ]:
with open('lit_pocket_encoded_ep3.pkl', 'wb') as p:
    pickle.dump(prot_encoded, p)
with open('lit_ligand_encoded_ep3.pkl', 'wb') as p:
    pickle.dump(lig_encoded, p)